### Load Results

In [1]:
import json
import pandas as pd
from pathlib import Path
from statsmodels.stats.inter_rater import fleiss_kappa
import numpy as np


results_path = Path("determinism")
df = pd.DataFrame()

for folder in results_path.iterdir():
    if not folder.is_dir(): continue
    for file in folder.iterdir():
        if file.suffix != ".csv": continue
        run = pd.read_csv(file)
        run["Language"] = folder.name
        run["Iteration"] = file.stem.split("_")[3]
        df = pd.concat([df, run], ignore_index=True)

df["Accuracy"] = df["LLM Hit CWE"] & df["LLM Hit Code"]

### Group DF by Results

In [32]:
grouped_df = df.groupby(["Language", "Test Case Weakness", "Test Case File"]).agg(
    **{
        "Hit": ("Accuracy", "sum"),
        "Count": ("Accuracy", "count")
    }
).reset_index()

grouped_df["No_Hit"] = grouped_df["Count"] - grouped_df["Hit"]
grouped_df["kappa"] = 0.0

conteos = []
for row in grouped_df.itertuples():
    conteo_0 = row.No_Hit
    conteo_1 = row.Hit
    conteos.append([conteo_0, conteo_1])
conteos = np.array(conteos)
Kappa = fleiss_kappa(conteos)
print(f"Kappa: {Kappa}")

frecuency = grouped_df.groupby(["Hit", "No_Hit"]).size().reset_index(name="Count")
frecuency["precision"] = abs(frecuency["Hit"] - frecuency["No_Hit"])
frecuency = frecuency.sort_values(["precision", "Count"], ascending=False)
frecuency

Kappa: 0.7067766106284189


,Hit,No_Hit,Count,precision
14,15,0,146,15
0,0,15,10,15
1,1,14,4,13
13,14,1,3,13
12,13,2,4,11
2,2,13,3,11
3,3,12,3,9
11,12,3,3,9
4,4,11,6,7
10,11,4,2,7


In [34]:
grouped_php_df = grouped_df[grouped_df["Language"] == "php"]
conteos = []
for row in grouped_php_df.itertuples():
    conteo_0 = row.No_Hit
    conteo_1 = row.Hit
    conteos.append([conteo_0, conteo_1])
conteos = np.array(conteos)
Kappa_php = fleiss_kappa(conteos)
print(f"Kappa PHP: {Kappa_php}")

php_frecuency = grouped_php_df.groupby(["Hit", "No_Hit"]).size().reset_index(name="Count")
php_frecuency["precision"] = abs(php_frecuency["Hit"] - php_frecuency["No_Hit"])
php_frecuency = php_frecuency.sort_values(["precision", "Count"], ascending=False)
php_frecuency

Kappa PHP: 0.6849158490679877


,Hit,No_Hit,Count,precision
13,15,0,64,15
0,0,15,7,15
1,1,14,2,13
12,14,1,2,13
2,2,13,3,11
11,13,2,3,11
3,3,12,2,9
10,12,3,2,9
4,4,11,2,7
9,11,4,1,7


In [35]:
grouped_cs_df = grouped_df[grouped_df["Language"] == "csharp"]
conteos = []
for row in grouped_cs_df.itertuples():
    conteo_0 = row.No_Hit
    conteo_1 = row.Hit
    conteos.append([conteo_0, conteo_1])
conteos = np.array(conteos)
Kappa_csharp = fleiss_kappa(conteos)
print(f"Kappa CSharp: {Kappa_csharp}")

csharp_frecuency = grouped_cs_df.groupby(["Hit", "No_Hit"]).size().reset_index(name="Count")
csharp_frecuency["precision"] = abs(csharp_frecuency["Hit"] - csharp_frecuency["No_Hit"])
csharp_frecuency = csharp_frecuency.sort_values(["precision", "Count"], ascending=False)
csharp_frecuency


Kappa CSharp: 0.7275837351125638


,Hit,No_Hit,Count,precision
11,15,0,82,15
0,0,15,3,15
1,1,14,2,13
10,14,1,1,13
9,13,2,1,11
2,3,12,1,9
8,12,3,1,9
3,4,11,4,7
7,11,4,1,7
6,10,5,2,5


### Export Results to Excel

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("determinism_results.csv", index=False)